In [ ]:
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
import warnings
warnings.filterwarnings("ignore")

import sys
#import os
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py as h5
import sklearn
from sklearn.multioutput import MultiOutputClassifier
from sklearn import metrics
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, precision_recall_curve, accuracy_score, confusion_matrix  
from sklearn.metrics import average_precision_score
import pickle
import keras
from keras.models import load_model
import sys
sys.path.append(r'C:\Users\aoara\develop\deepbeat')
import utils
from pathlib import Path
from tensorflow.keras.optimizers import Adam
from scipy.io import loadmat
import copy
#import seaborn as sns


# Load model

In [ ]:
import argparse
import json
import h5py
from pathlib import Path


h5_file_path = r'C:\develop\afib_detection\keras\deepbeat.h5'
config = None
with h5py.File(h5_file_path, 'r') as f:
    training_config = json.loads(f.attrs['training_config'])
    optimizer_config = training_config['optimizer_config']
    
print(training_config)
print(optimizer_config)

{'optimizer_config': {'class_name': 'Adam', 'config': {'lr': 3.906250185536919e-06, 'beta_1': 0.8999999761581421, 'beta_2': 0.9990000128746033, 'decay': 0.0, 'epsilon': 1e-07, 'amsgrad': False}}, 'loss': {'qa_output': 'categorical_crossentropy', 'rhythm_output': 'binary_crossentropy'}, 'metrics': ['acc'], 'sample_weight_mode': None, 'loss_weights': [0.2, 5.0]}
{'class_name': 'Adam', 'config': {'lr': 3.906250185536919e-06, 'beta_1': 0.8999999761581421, 'beta_2': 0.9990000128746033, 'decay': 0.0, 'epsilon': 1e-07, 'amsgrad': False}}


In [ ]:
def get_training_config():
    with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
        if 'keras_version' in f.attrs:
            print(f"Keras version: {f.attrs['keras_version']}")
        
        if 'backend' in f.attrs:
            print(f"Backend: {f.attrs['backend']}")
        
        # Sometimes stored under model config
        if 'model_config' in f.attrs:
            import json
            config = json.loads(f.attrs['model_config'])
            if 'keras_version' in config:
                print(f"Keras version (from config): {config['keras_version']}")
    
    training_config = json.loads(f.attrs['training_config'])
    return training_config 

In [ ]:
# get training config
with h5.File(r"C:\Users\aoara\develop\deepbeat\deepbeat.h5", 'r') as f:
    # Check for version attributes
    if 'keras_version' in f.attrs:
        print(f"Keras version: {f.attrs['keras_version']}")
    
    if 'backend' in f.attrs:
        print(f"Backend: {f.attrs['backend']}")
    
    # Sometimes stored under model config
    if 'model_config' in f.attrs:
        import json
        config = json.loads(f.attrs['model_config'])
        if 'keras_version' in config:
            print(f"Keras version (from config): {config['keras_version']}")
    
    # Print all available attributes
    print("\nAll attributes in file:")
    for key in f.attrs.keys():
        print(f"  {key}: {f.attrs[key]}")
    
    training_config = json.loads(f.attrs['training_config'])

In [ ]:
orig_config = training_config ['optimizer_config']['config']
orig_config['learning_rate'] = orig_config.pop('lr') # rename lr to learning rate
orig_config.pop('decay') # there is no longer a parameter called decay; the original decay was 0

# load deepbeat model with new tensorflow package, verify performances
path_to_model =r'C:\Users\aoara\develop\deepbeat'
model_name = 'deepbeat.h5'
deepbeat = load_model( Path(path_to_model) / model_name, compile = False) 

## Verify Loaded Model Performances

In [ ]:
## load original test data
data_path = Path(r'C:\Users\aoara\develop\deepbeat\data\db')
data_test = np.load(data_path / 'test.npz', allow_pickle=True)
test_x = data_test['signal']
test_qa = data_test['qa_label']
test_r = data_test['rhythm']
test_p = pd.DataFrame(data_test['parameters'])
print('test data shape: ')
print(test_x.shape)
print(test_qa.shape)
print(test_r.shape)
print(test_p.shape)
test_p.rename(index=str, columns={0:'timestamp', 
                                  1:'stream', 
                                  2:'ID'}, inplace=True)
## QA results

predictions_qa, predictions_r = deepbeat.predict(test_x)
predictions_QA = np.argmax(predictions_qa, axis=1)

#print(classification_report(np.argmax(test_qa, axis=1), predictions_QA))


excellent_qa_indx = np.where(predictions_QA==2)[0]
x_test_excellent = test_x[excellent_qa_indx,:]
p_test_excellent = test_p.iloc[excellent_qa_indx,:]
rhythm_test_excellent = test_r[excellent_qa_indx,:]
quality_assessment_test_excellent = test_qa[excellent_qa_indx,:]


# weighted macro-average across all indivduals

test_metrics_1 = utils.collecting_individual_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
test_metrics = pd.DataFrame.from_dict(test_metrics_1).T.rename(columns={0:'TPR', 1:'TNR', 2:'FPR', 3:'FNR', 4:"total_samples"})


for m in ['TPR', 'TNR', 'FPR', 'FNR']:
    metric_wmu = np.average(test_metrics[m][~test_metrics[m].isna()], weights=test_metrics['total_samples'][~test_metrics[m].isna()])
    print('%s: %0.2f' % (m, metric_wmu))
    
# PPV, NPV and F1

episode_m = utils.episode_metrics(deepbeat, x_test_excellent, p_test_excellent, rhythm_test_excellent, out_message=False)
episode_metrics = pd.DataFrame(episode_m).T
episode_metrics.rename(columns={0:'TPR', 1:'TNR', 2:'PPV', 3:'NPV', 4:"FPR", 5:'FNR', 6:'F1', 7:'total_samples'}, inplace=True)
for m in ['PPV', 'NPV', 'F1']:
    print('%s: %0.2f' % (m, episode_metrics[m]))